# Работа с табличными данными

## 1. Pandas

**Pandas** — это одна из самых популярных библиотек Python для работы с табличными данными. Она предоставляет удобные 
структуры данных:
* **Series** — одномерный массив с индексами, похожий на колонку в таблице;
* **DataFrame** — двумерная таблица, где каждая колонка может иметь свой тип данных.

![pandas](../misc/images/pandas.jpeg)

С помощью Pandas можно: фильтровать строки и столбцы, группировать данные, агрегировать показатели, объединять таблицы, 
работать с пропущенными значениями и строить простые визуализации. Владение Pandas — базовый и необходимый навык для 
проведения EDA.

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Задание 1.1. Загрузка данных
1. Пройди по ссылке на хранилище с датасетом [2025 Kaggle Machine Learning & Data Science Survey](https://disk.360.yandex.ru/d/ATLDXjTVorMf3Q).
3. Скачай CSV-файлы с данными и сохрани их в папку `datasets`.
4. С помощью библиотеки `pandas` загрузи:
   * таблицу с выбором ответов (multiple choice) в переменную `multiple`,
   * таблицу со свободными ответами (free form) `freeform`.
5. Выведи размеры каждой таблицы.
   
> **Важно:** загружать датасеты на Git не нужно.

### Задание 1.2. Словарь вопросов
В таблицах `freeform` и `multiple` первая строка содержит текстовые формулировки вопросов (например: *'What is your gender? — Selected Choice'*, *'What is your age (# years)?'* и т. д.), а сами данные начинаются со второй строки.

1. Создай структуру `name2question`, которая для каждой колонки содержит словарь с ключами `question` и `type`. В ключе `question` хранится текст вопроса, а в ключе `type` — тип таблицы, из которой пришел вопрос: `'multiple'`, `'freeform'` или `'both'` (если колонка есть в обеих таблицах).
```
name2question['Q1']
{'question': 'What is your gender? - Selected Choice', 'type': 'multiple'}
```

2. Выведи значение для вопроса `Q1_OTHER_TEXT` из структуры `name2question`.
3. Удали первые строки из таблиц `freeform` и `multiple`.
4. Выведи размерности таблиц `freeform` и `multiple`.

### Задание 1.3. Анализ общих колонок

В таблице `multiple` некоторые колонки содержат числовые коды ответов. Если значение отлично от `-1`, это означает, что респондент выбрал свой вариант ответа, и в таблице `freeform` для той же строки (с тем же индексом) должно присутствовать соответствующее текстовое значение в свободной форме.

Проверить возможность корректного объединения таблиц `multiple` и `freeform`.

1. Определи общие колонки:
   - Найди все колонки, которые присутствуют одновременно в таблицах `multiple` и `freeform`.
   - Выведи их количество.

2. Проведи анализ общих колонок:
   - Выбери 2–3 общие колонки для детального изучения (например, `Q1_OTHER_TEXT`, `Q6_OTHER_TEXT`).
   - Для каждой выбранной колонки:
     - Выведи частоты значений в таблице `multiple` (количество значений, отличных от `-1`).
     - Выведи частоты значений в таблице `freeform` (количество непустых текстовых значений).
     - Найди индексы строк, где в таблице `multiple` значение отлично от `-1`.
     - Найди индексы строк, где в таблице `freeform` присутствуют текстовые значения.
     - Сравни эти два набора индексов.

3. Напиши свои выводы по результатам анализа:
   - Можно ли корректно объединить эти две таблицы по индексам строк?
   - Если нет, приведи конкретные примеры, демонстрирующие проблему (выведи соответствующие строки из обеих таблиц).

***Результаты анализа:***

### Задание 1.4. Объединение таблиц
В предыдущем задании мы выяснили, что таблицы `multiple` и `freeform` нельзя просто объединить по номерам строк — данные не совпадают. Но мы можем попробовать восстановить связи между числовыми кодами из `multiple` и текстовыми ответами из `freeform`.

_Идея:_ если какое-то текстовое значение встречается в `freeform` определенное количество раз, и в `multiple` есть числовой код, который встречается такое же количество раз, причем эта частота уникальна (встречается только один раз в каждой таблице), то, скорее всего, это одно и то же значение.

Что нужно сделать:
1. Создай копию таблицы `multiple` и сохрани ее в переменную `responses`.
2. Для каждой общей колонки:
   - Посчитай, сколько раз встречается каждое значение в `multiple` (не считай `-1` и пустые значения).
   - Посчитай, сколько раз встречается каждое значение в `freeform` (не считай пустые строки и пустые значения).
   - Найди пары, где:
     * Частота совпадает (например, и код, и текст встречаются по 7 раз).
     * Эта частота уникальна (встречается только один раз в каждой таблице).
   - Выполни замену значений в колонке таблицы `responses`:
     * Значения, которые были равны `-1`, оставь без изменений (`-1`).
     * Найденные числовые коды (для которых удалось найти соответствующее текстовое значение) замени на соответствующие текстовые значения из `freeform`.
     * Для всех остальных числовых кодов (которые не удалось сопоставить) замени на `-2`.
3. Подсчитай и выведи:
   - Средний процент текстовых значений из `freeform`, которые удалось успешно сопоставить с числовыми кодами из `multiple`, по всем колонкам **(в процентах)**.
   - Название колонки, для которой получилось сопоставить больше всего значений.


### Задание 1.5. Анализ пропусков
Мы объединили наши данные ответов в сводной форме. Давай проанализируем, насколько респондентам хватало предложенных вариантов ответа на вопросы. В опросе есть вопросы, где респонденты могли выбрать вариант из предложенных, а также написать свой ответ в свободной форме. Если респондент часто предпочитал сам давать ответы, это может указывать на то, что предложенные варианты были недостаточно полными.

1. Найди все **вопросы**, в которых встречаются колонки со свободным ответом. Исключи вопрос `Q12` — в нем свободный ответ встречается несколько раз. Выведи количество таких вопросов.

2. Среди этих вопросов найди:

   - Вопрос, на который респонденты чаще всего не выбирали ответа из предложенных.
   - Вопрос, на который респонденты всегда выбирали ответ из предложенного.
   - Вопрос, на который респонденты чаще всего писали свободный ответ.

### Задание 1.6. Очистка респондентов

В данных есть информация о времени заполнения опроса. Логично предположить, что, если респондент отвечал **слишком быстро** или, наоборот, **слишком долго**, такие ответы могут быть недостоверными. Для очистки данных воспользуемся методом выявления выбросов на основе **box plot**.

**Box plot (ящик с усами)** — это способ визуализации распределения данных. Внутри прямоугольника (ящика) находятся значения от 1-го до 3-го квартиля (Q1 и Q3), линия внутри — это медиана. «Усы» обычно определяются через **интерквартильный размах (IQR)**, который равен:

$$IQR = Q3 - Q1$$
Значения ниже $Q1 - 1.5 \times IQR$ и выше $Q3 + 1.5 \times IQR$ считаются выбросами.

<center><img src="../misc/images/box-plot.png" alt="analytics" width="600" height="400">

1. Построй box plot для времени заполнения опроса. Ограничь ось **X** от 0 до 5000 секунд.
2. Рассчитай интерквартильный размах (IQR).
3. Выведи нижнюю и верхнюю границы интервала для выявления выбросов.
4. Так как нижняя граница получилась отрицательной, будем априорно считать минимальное адекватное время заполнения равным **2 минутам (120 секундам)**.
5. Отфильтруй DataFrame, оставив только респондентов с временем заполнения от 120 секунд до верхней границы.
6. Выведи размерность очищенного датасета.

### Задание 1.7. Второй язык
1. Проанализируй, какие языки программирования чаще всего используют респонденты в каждой профессии. Используй `groupby` и посмотри список языков программирования внутри каждой группы.
2. Для роли `Statistician` назови самый популярный язык.
3. Для роли `Student` укажи самый непопулярный язык.
4. Для роли `Data Engineer` найди второй по популярности язык.
5. Для роли `Data Scientist` определи пятый по популярности язык.


### Задание 1.8. Кто быстрее
Давай посмотрим, кто быстрее отвечает на вопросы в опросе в разрезе пол/возраст. Это может помочь понять, есть ли различия в подходе к заполнению опроса между разными группами респондентов.

Составь сводную таблицу в разрезе пол/возраст по среднему времени заполнения опроса. Рассматривай только респондентов с полом `Male` и `Female`.

1. Отфильтруй данные, оставив только респондентов с полом `Male` и `Female`.
2. Создай сводную таблицу, где:
   - строки — пол,
   - столбцы — возрастные группы,
   - значения — среднее время заполнения опроса.
3. Покрась зеленым цветом ячейку для каждого возрастного промежутка, где показано, кто быстрее заполнил опросник (мужчины или женщины).
4. Выведи полученную сводную таблицу и ее размерность.


### Бонусное задание. Инсайды

Ты уже освоил основные инструменты для работы с данными: фильтрацию, группировку и создание сводных таблиц. Теперь настало время проявить творческий подход и найти собственные инсайты в данных!

**Задача:** самостоятельно придумай и исследуй **2 интересных вопроса** (инсайта), которые ты хочешь узнать из данных опроса Kaggle. Это могут быть вопросы о:
- различиях между профессиями,
- зависимости навыков от опыта,
- географических особенностях,
- связи между инструментами и ролями,
- или любые другие закономерности, которые тебе интересны.

**Требования:**
1. Сформулируй **2 вопроса**, которые ты хочешь проверить на данных.
2. Для проверки:
   - Используй **фильтрацию**, **сводную таблицу** или **группировку** (или их комбинацию).
   - Выведи результаты анализа.
   - Сделай краткий вывод: что ты узнал из данных?

**Примеры вопросов для вдохновения:**
- В каких странах Data Scientists зарабатывают больше всего?
- Какие инструменты используют респонденты с опытом более 10 лет?
- Есть ли связь между образованием и зарплатой в разных ролях?

<img src="../misc/images/analytics.png" alt="analytics" width="600" height="400">

> **Важно:** это бонусное задание, поэтому ты можешь проявить креативность и исследовать то, что тебе действительно интересно!
